# Step 1 — Freeze protocol + offline scoring harness

**Claim track:** noise-aware gated SASV (no local mic data).

## Protocol (locked)
- Corpus: **ASVspoof 2019 LA**
- Trials: official **SASV 2022** GI lists (`SASVC2022_Baseline`)
- Tune on **dev** only; report **eval** once
- Metrics: **SASV-EER**, **SV-EER**, **SPF-EER**

## This notebook
Prints paths, verifies protocol files, and writes a **smoke** ECAPA-only score CSV
(offline — not the Docker demo).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name != "sasv_noise_gated":
    # Allow running from repo root
    cand = ROOT / "replay-cnn-baseline" / "experiments" / "sasv_noise_gated"
    if cand.exists():
        ROOT = cand
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent / "sasv_la2019"))

from noise_gated_lib import (
    DEFAULT_LA,
    DEFAULT_SASV,
    ensure_dirs,
    ensure_sasv_on_path,
    protocol_summary,
    read_trials,
    save_json,
    trial_key_counts,
    RUNS_DIR,
)

ensure_dirs()
proto = protocol_summary()
proto

### Check data layout

If any path is missing, clone/download before full runs (see folder `README.md`).

In [ ]:
from pathlib import Path

print("LA exists:", Path(DEFAULT_LA).exists(), DEFAULT_LA)
print("SASV baseline exists:", Path(DEFAULT_SASV).exists(), DEFAULT_SASV)

sasv = ensure_sasv_on_path(DEFAULT_SASV)
for split in ("dev", "eval"):
    trials = read_trials(sasv, split, max_trials=None)
    print(split, "trials:", len(trials), trial_key_counts(trials))

### Offline harness smoke (ECAPA-only, 200 trials on dev)

Uses `sasv_la2019.score_lib` — same metrics as the locked SASV notebooks.
Set `SMOKE = False` only when you intend a longer run.

In [ ]:
SMOKE = True
MAX_TRIALS = 200 if SMOKE else 0  # 0 = all trials
DEVICE = "cuda"  # falls back to cpu inside score_lib if needed
FORCE_CPU = False

from score_lib import score_ecapa_trials

out = RUNS_DIR / "harness_smoke_ecapa_dev"
summary = score_ecapa_trials(
    la_root=DEFAULT_LA,
    sasv_root=DEFAULT_SASV,
    split="dev",
    max_trials=MAX_TRIALS,
    device=DEVICE,
    force_cpu=FORCE_CPU,
    output_dir=out,
)
save_json(out / "protocol.json", proto)
summary

### Done when
- Protocol paths resolve
- A `scores_*.csv` + `metrics_*.json` appear under `runs/harness_smoke_ecapa_dev/`

Next → **02_noise_injection.ipynb**